# NB6 (Colab GPU) — PVSetFormer's test rows A4c, A2 and the test bed's main

D80, after the protocol freeze (frozen configuration `2d27ea1e82060a0f…`). The laptop's 4 cores would need about 6 days for these rows; the main row, A6e and A6d stay on its CPU. Same model, inputs, settings and seeds; only the device differs. First a validation round (V12 on the 10 validation units, the 5 frozen seeds) and its gate: the GPU ensemble must be level with the CPU one (under 3% at every horizon) and the per-seed medians within 3 standard errors. Only then the test rows.

**How to run:** Runtime > Change runtime type > a GPU. Then Runtime > Run all. Keep the tab open. If Colab disconnects, reconnect and Run all again: finished trainings are kept on Drive and skipped. The last cell unmounts Drive. Then, on the laptop: `python tools/fetch_drive_runs.py --set nb6gpu`.

In [ ]:
import os, sys, json, hashlib, shutil, subprocess, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DRIVE_REPO = Path('/content/drive/MyDrive/pv-cold-start-nb6gpu')
ROOT = Path('/content/repo')
man = json.loads((DRIVE_REPO / 'MIRROR_MANIFEST.json').read_text())
if ROOT.exists():
    shutil.rmtree(ROOT)          # links are removed, not followed: Drive keeps the outputs
for f, h in man['sha256'].items():
    d = ROOT / f
    d.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(DRIVE_REPO / f, d)
    assert hashlib.sha256(d.read_bytes()).hexdigest() == h, f'{f} differs from the mirror manifest'
shutil.copyfile(DRIVE_REPO / 'MIRROR_MANIFEST.json', ROOT / 'MIRROR_MANIFEST.json')
for rel in ['results/nb6_forecasts/parts', 'results/nb6_gpu_val']:   # outputs live on Drive
    (DRIVE_REPO / rel).mkdir(parents=True, exist_ok=True)
    (ROOT / rel).parent.mkdir(parents=True, exist_ok=True)
    os.symlink(DRIVE_REPO / rel, ROOT / rel)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pvlib==0.16.1'], check=True)   # the gate's priors
import torch, numpy, pandas
assert torch.cuda.is_available(), 'no GPU: Runtime > Change runtime type > a GPU'
ENV = {**os.environ, 'PVSF_DEVICE': 'cuda', 'CUBLAS_WORKSPACE_CONFIG': ':4096:8', 'PYTHONUNBUFFERED': '1'}
PROCS = max(1, min(4, os.cpu_count()))   # trainings at once on the GPU (one CPU each)
print('mirror commit', man['commit'], '| files', len(man['sha256']), '| GPU', torch.cuda.get_device_name(0),
      '| CPUs', os.cpu_count(), '| processes', PROCS, '| torch', torch.__version__, '| numpy', numpy.__version__,
      '| pandas', pandas.__version__)

def step(*args):
    t = time.time()
    p = subprocess.Popen([sys.executable, '-u', 'tools/nb6_gpu.py', *args], cwd=ROOT, env=ENV,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    assert p.wait() == 0, f'{args} failed (see above)'
    print(f'--- {" ".join(args)} done in {(time.time() - t) / 60:.1f} min', flush=True)

Validation round (150 trainings), then the gate. The gate stops the notebook if the GPU is not level with the CPU.

In [ ]:
step('--validate', '--procs', str(PROCS))
step('--gate')

The test rows, longest trainings first. Each row prints its mean training time.

In [ ]:
for row in ['A4c', 'A2', 'tb_main']:
    step('--row', row, '--procs', str(PROCS))

In [ ]:
drive.flush_and_unmount()
print('done: on the laptop run python tools/fetch_drive_runs.py --set nb6gpu')